# NBA 球员数据监督学习 —— XGBoost 进阶模型 vs Baseline 对比

本 notebook 完全复用 `03_prediction_baseline.ipynb` 中的数据预处理与时间切分逻辑，只新增 **XGBoost** 进阶模型，并用相同测试集指标与 Baseline 做直观对比。

Baseline 参考指标（取自 `03_prediction_baseline.ipynb` 测试集输出）：
- 回归：`LinearRegression` R²=0.3441、RMSE=4.7785；`DecisionTreeRegressor` R²=0.2839、RMSE=4.9930；
- 分类：`LogisticRegression` F1=0.7401、Recall=0.6142。

> 本阶段只输出指标对比，不绘制 SHAP 图、不导出 `.pkl` 模型。

## 1. 数据准备与读取（与 Baseline 一致）

### 1.1 导入依赖库

In [1]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier, XGBRegressor

print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"xgboost      : {xgb.__version__}")

pandas       : 2.3.3
scikit-learn : 1.9.0
xgboost      : 2.1.4


### 1.2 读取主数据集

In [2]:
DATA_PATH = "../data/processed/master_data.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"主数据集规模: {df_raw.shape[0]} 行 x {df_raw.shape[1]} 列")
print(f"赛季覆盖范围: {df_raw['Year'].min()} - {df_raw['Year'].max()}")

主数据集规模: 20313 行 x 78 列
赛季覆盖范围: 1950 - 2017


### 1.3 构造回归标签 `PER_next` 并剔除无关 / 易泄露字段

`PER_next` 需要在删除 `Player` 前按球员分组完成 `shift(-1)`，之后统一剔除身份标识、球队环境与出生信息字段。

In [3]:
# 按 (Player, Year) 排序后，分组移位得到下一赛季 PER
df = df_raw.sort_values(["Player", "Year"]).reset_index(drop=True)
df["PER_next"] = df.groupby("Player")["PER"].shift(-1)

# 剔除字段与 Baseline notebook 保持一致
drop_cols = [
    "Player",
    "Tm",
    "college",
    "birth_year",
    "birth_city",
    "birth_state",
    "birth_date",
    "year_start",
    "year_end",
    "position_career",
]

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

print(f"剔除后数据规模: {df_model.shape[0]} 行 x {df_model.shape[1]} 列")

剔除后数据规模: 20313 行 x 69 列


### 1.4 Pos 独热编码与公共统计特征

使用与 Baseline 完全相同的特征集合：11 项当季统计特征 + `Pos` 独热哑变量。

In [4]:
feature_cols = [
    "Age",
    "G",
    "MP",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "PTS_per36",
    "AST%",
    "TRB%",
    "WS",
]

pos_dummies = pd.get_dummies(df_model["Pos"], prefix="Pos").astype(int)
pos_cols = pos_dummies.columns.tolist()

df_model = pd.concat(
    [df_model.drop(columns=["Pos"]), pos_dummies],
    axis=1,
)

print(f"Pos 独热编码后新增 {len(pos_cols)} 个哑变量列")
print(f"统计特征: {feature_cols}")

Pos 独热编码后新增 23 个哑变量列
统计特征: ['Age', 'G', 'MP', 'TS%', '3PAr', 'FTr', 'USG%', 'PTS_per36', 'AST%', 'TRB%', 'WS']


## 2. 构造两任务的标签 (y) 与特征矩阵 (X)

In [5]:
def build_X(data: pd.DataFrame) -> pd.DataFrame:
    """构造两任务共用的当季统计特征矩阵（统计特征 + Pos 哑变量）。"""
    return data[feature_cols + pos_cols].copy()

### 2.1 回归任务：预测下赛季 `PER_next`

In [6]:
# 剔除无法匹配到下一年数据的记录
reg_df = df_model[df_model["PER_next"].notna()].copy()

X_reg = build_X(reg_df)
y_reg = reg_df["PER_next"]
year_reg = reg_df["Year"]

print(f"回归样本数: {X_reg.shape[0]}")
print(f"X_reg 维度 : {X_reg.shape}")

回归样本数: 16260
X_reg 维度 : (16260, 34)


### 2.2 分类任务：当季是否达到全明星实力门槛

标签规则与 Baseline 一致：`PER >= 20.0` 且 `WS >= 6.0` 时为 1，否则为 0。

In [7]:
# 标签依赖 PER 与 WS，缺失时无法可靠判定，先剔除
cls_df = df_model.dropna(subset=["PER", "WS"]).copy()

cls_df["Is_AllStar_Caliber"] = (
    (cls_df["PER"] >= 20.0) & (cls_df["WS"] >= 6.0)
).astype(int)

X_cls = build_X(cls_df)
y_cls = cls_df["Is_AllStar_Caliber"]
year_cls = cls_df["Year"]

print(f"分类样本数: {X_cls.shape[0]}")
print(f"X_cls 维度 : {X_cls.shape}")
print(f"正类占比  : {y_cls.mean():.4%}")

分类样本数: 19921
X_cls 维度 : (19921, 34)
正类占比  : 5.3009%


## 3. 时间维度切分（Train: `Year <= 2010` / Test: `Year > 2010`）

In [8]:
def time_split(X, y, years, train_max=2010):
    """按赛季年份切分：训练集取 years <= train_max，测试集取 years > train_max。"""
    train_mask = years <= train_max
    test_mask = ~train_mask
    return (
        X.loc[train_mask],
        X.loc[test_mask],
        y.loc[train_mask],
        y.loc[test_mask],
    )


# 回归任务
X_train_reg, X_test_reg, y_train_reg, y_test_reg = time_split(X_reg, y_reg, year_reg)
# 分类任务
X_train_cls, X_test_cls, y_train_cls, y_test_cls = time_split(X_cls, y_cls, year_cls)

print("【回归】训练集:", X_train_reg.shape)
print("【回归】测试集:", X_test_reg.shape)
print()
print("【分类】训练集:", X_train_cls.shape)
print("【分类】测试集:", X_test_cls.shape)
print()
print(
    f"分类正类占比 -> 训练集: {y_train_cls.mean():.4%} | 测试集: {y_test_cls.mean():.4%}"
)

【回归】训练集: (13883, 34)
【回归】测试集: (2377, 34)

【分类】训练集: (16587, 34)
【分类】测试集: (3334, 34)

分类正类占比 -> 训练集: 5.1788% | 测试集: 5.9088%


## 4. XGBoost 进阶模型评估与 Baseline 对比

### 4.1 回归任务：XGBRegressor vs Baseline

- 模型：`XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)`；
- 预处理：与 Baseline 保持一致的中位数填补（树模型无需标准化）；
- 评估：测试集上的 R² 与 RMSE。

In [9]:
# Baseline 回归指标（来自 03_prediction_baseline.ipynb 的测试集输出）
baseline_reg = [
    {"Model": "LinearRegression", "R2": 0.3441, "RMSE": 4.7785},
    {"Model": "DecisionTreeRegressor", "R2": 0.2839, "RMSE": 4.9930},
]

xgb_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
    )),
])

xgb_reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = xgb_reg_pipeline.predict(X_test_reg)

r2_reg = r2_score(y_test_reg, y_pred_reg)
rmse_reg = float(np.sqrt(mean_squared_error(y_test_reg, y_pred_reg)))

print("[回归] XGBRegressor")
print(f"  R2   = {r2_reg:.4f}")
print(f"  RMSE = {rmse_reg:.4f}")
print()

# 与 Baseline 拼接成对比表格
comparison_reg = pd.DataFrame(
    baseline_reg + [{"Model": "XGBRegressor", "R2": r2_reg, "RMSE": rmse_reg}]
)
print("回归指标对比表:")
print(comparison_reg.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

[回归] XGBRegressor
  R2   = 0.3494
  RMSE = 4.7591

回归指标对比表:
                Model     R2   RMSE
     LinearRegression 0.3441 4.7785
DecisionTreeRegressor 0.2839 4.9930
         XGBRegressor 0.3494 4.7591


### 4.2 分类任务：XGBClassifier vs Baseline

- 针对正负样本不平衡（测试集正类约 5.9%），设置 `scale_pos_weight=15.6`；
- 模型：`XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)`；
- 评估：测试集上的 Accuracy / Precision / Recall / F1-Score。

In [10]:
# Baseline 分类指标（完整四类指标取自 03_prediction_baseline.ipynb 测试集输出）
baseline_cls = {
    "Accuracy": 0.9745,
    "Precision": 0.9308,
    "Recall": 0.6142,
    "F1-Score": 0.7401,
}

xgb_cls_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=15.6,
        random_state=42,
    )),
])

xgb_cls_pipeline.fit(X_train_cls, y_train_cls)
y_pred_cls = xgb_cls_pipeline.predict(X_test_cls)

acc_cls = accuracy_score(y_test_cls, y_pred_cls)
prec_cls = precision_score(y_test_cls, y_pred_cls)
recall_cls = recall_score(y_test_cls, y_pred_cls)
f1_cls = f1_score(y_test_cls, y_pred_cls)

print("[分类] XGBClassifier")
print(f"  Accuracy  = {acc_cls:.4f}")
print(f"  Precision = {prec_cls:.4f}")
print(f"  Recall    = {recall_cls:.4f}")
print(f"  F1-Score  = {f1_cls:.4f}")
print()

comparison_cls = pd.DataFrame([
    {
        "Model": "LogisticRegression",
        "Accuracy": baseline_cls["Accuracy"],
        "Precision": baseline_cls["Precision"],
        "Recall": baseline_cls["Recall"],
        "F1-Score": baseline_cls["F1-Score"],
    },
    {
        "Model": "XGBClassifier",
        "Accuracy": acc_cls,
        "Precision": prec_cls,
        "Recall": recall_cls,
        "F1-Score": f1_cls,
    },
])
print("分类指标对比表:")
print(comparison_cls.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

[分类] XGBClassifier
  Accuracy  = 0.9796
  Precision = 0.7654
  Recall    = 0.9442
  F1-Score  = 0.8455

分类指标对比表:
             Model  Accuracy  Precision  Recall  F1-Score
LogisticRegression    0.9745     0.9308  0.6142    0.7401
     XGBClassifier    0.9796     0.7654  0.9442    0.8455


### 4.3 综合对比结论

使用与 Baseline **完全相同的测试集与评估指标**，计算 XGBoost 相对最优 Baseline 的指标差值，判断是否有显著改善。

In [11]:
# 以数值指标最优的 LinearRegression 作为回归 Baseline
lr_r2, lr_rmse = baseline_reg[0]["R2"], baseline_reg[0]["RMSE"]
lr_recall, lr_f1 = baseline_cls["Recall"], baseline_cls["F1-Score"]

r2_gain = r2_reg - lr_r2
rmse_gain = lr_rmse - rmse_reg
recall_gain = recall_cls - lr_recall
f1_gain = f1_cls - lr_f1

print("=" * 72)
print("回归对比: XGBRegressor vs LinearRegression (Baseline)")
print(f"  R2   : {r2_reg:.4f} vs {lr_r2:.4f}  -> 提升 {r2_gain:+.4f}")
print(f"  RMSE : {rmse_reg:.4f} vs {lr_rmse:.4f}  -> 差值 {rmse_gain:+.4f} (正值表示 RMSE 降低)")
print()
print("分类对比: XGBClassifier vs LogisticRegression (Baseline)")
print(f"  Recall: {recall_cls:.4f} vs {lr_recall:.4f}  -> 提升 {recall_gain:+.4f}")
print(f"  F1    : {f1_cls:.4f} vs {lr_f1:.4f}  -> 提升 {f1_gain:+.4f}")
print("=" * 72)

# 结论判定：R2 更高、RMSE 更低视为回归更优；Recall/F1 更高视为分类更优
reg_better = (r2_gain > 0) and (rmse_gain > 0)
cls_better = (recall_gain > 0) and (f1_gain > 0)

print(f"回归结论: XGBRegressor 相对 Baseline {'更优' if reg_better else '未更优'} (R2 {r2_reg:.4f}, RMSE {rmse_reg:.4f})")
print(f"分类结论: XGBClassifier 相对 Baseline {'更优' if cls_better else '未更优'} (Recall {recall_cls:.4f}, F1 {f1_cls:.4f})")

回归对比: XGBRegressor vs LinearRegression (Baseline)
  R2   : 0.3494 vs 0.3441  -> 提升 +0.0053
  RMSE : 4.7591 vs 4.7785  -> 差值 +0.0194 (正值表示 RMSE 降低)

分类对比: XGBClassifier vs LogisticRegression (Baseline)
  Recall: 0.9442 vs 0.6142  -> 提升 +0.3300
  F1    : 0.8455 vs 0.7401  -> 提升 +0.1054
回归结论: XGBRegressor 相对 Baseline 更优 (R2 0.3494, RMSE 4.7591)
分类结论: XGBClassifier 相对 Baseline 更优 (Recall 0.9442, F1 0.8455)


## 5. 阶段小结

- 回归任务通过 **R² / RMSE** 对比 XGBRegressor 与 LinearRegression / DecisionTreeRegressor；
- 分类任务通过 **Accuracy / Precision / Recall / F1** 对比 XGBClassifier 与 LogisticRegression；
- 本阶段未进行 SHAP 解释，也未导出 `.pkl` 模型，后续可在确认模型收益后再继续推进。